
LoRA(Low_Rank Adaptation)是一种参数高效微调方法
核心思想：在与训练模型的权重矩阵旁边插入两个低秩矩阵，通过这两个矩阵来适应下游任务，冻结原始的参数模型

线性LoRA特对线性层，把线性层的权重变化分解为低秩成积: $\bigtriangleup W = B \times A$
A表示输入维度，B表示输出维度，前向传播的时候:

$输出=原线性层输出 + (B \times A \times x)$


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
class LoRALinear(nn.Module):
    """
    带LoRA适配的线性层
    原始线性层: y = Wx + b
    LoRA线性层: y = Wx + b + (B @ A) @ β 
    矩阵运算法则: (A+B)C = AC + BC
    """
    def __init__(
        self,
        in_features: int,
        out_features: int,
        r:  int = 8, #LoRA的秩
        lora_alpha: int = 16, # 缩放因子（通常等于2*r）
        lora_dropout:  float = 0.0,
        merge_weight: bool = False,
        **kwargs
    ):
        super().__init__()
        self.in_features = in_features,
        self.out_features = out_features
        self.r =  r #LoRA的秩
        self.lora_alpha = lora_alpha
        self.lora_dropout = lora_dropout
        self.merge_weight = merge_weight

        # 1. 定义原线性层（冻结，不训练）
        self.linear = nn.Linear(in_features,out_features,**kwargs)

        # 2. 定义LoRA低秩矩阵
        self.lora_A  = nn.Linear(in_features, r, bias = False)
        self.lora_B  = nn.Linear(r, out_features, bias = False)

        # 3. Dropout
        self.lora_dropout = nn.Dropout(lora_dropout) if lora_dropout > 0 else nn.Identity()

        # 4. 初始化，A用高斯分布，B初始化为0
        nn.init.normal_(self.lora_A.weight,mean=0,std=1e-4)
        nn.init.zeros_(self.lora_B.weight)

        # 5. 标记，原linear权重冻结，只有LoRA部分可以训练
        self.linear.weight.requires_grad = False
        if self.linear.bias is not None:
            self.linear.bias.requires_grad  = False
    def merge_lora_weighes(self):
        if self.merge_weight:  #如果已经合并了就返回
            return
        delta_W = (self.lora_A.weight @ self.lora_B.weight)*(self.lora_alpha/self.r)
        self.linear.weight.data += delta_W
        self.merge_weight = True

    def unmerge_lora_weights(self):
        """解除LoRA权重合并（继续训练用）"""
        if not self.merge_weight:   #如果没有合并就返回
            return
        delta_W = (self.lora_A.weight @ self.lora_B.weight)*(self.lora_alpha/self.r)
        self.linear.weight.data -= delta_W
        self.merge_weight = False

    def forward(self,x:torch.Tensor) -> torch.Tensor:
        result = self.linear(x)
        if self.merge_weight:
            return result
        lora_out = self.lora_dropout(x)
        lora_out = self.lora_A(lora_out)
        lora_out = self.lora_B(lora_out)
        result += lora_out * (self.lora_alpha / self.r)


        
